[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/misda/blob/main/examples/comparative.ipynb)

# Static MISDA and PCA comparison

This notebook compares MISDA and PCA as reduced representations of the same original objective space. MISDA preserves selected original objectives; PCA uses transformed principal components. Their native diagnostics remain separate, while direct comparison uses a common leave-one-out reconstruction score at the same reduced dimension.

In [ ]:
from pathlib import Path
import subprocess
import sys

# In a repository checkout, test the local code. In Colab, install main.
target = ".[benchmarks]" if Path("pyproject.toml").exists() else "git+https://github.com/monacofj/misda.git@main#egg=misda[benchmarks]"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", target])


In [ ]:
import pandas as pd

import misda
import misda.benchmarks as bench

N = 500
SEED = 123

COMPARATIVE_CASES = (
    ("exp_01", bench.mopA_monotonic_redundancy),
    ("exp_02", bench.mopC_latent_blocks_4x5),
    ("exp_03", bench.mopD_pure_conflict_groups),
)

## Run the three comparative experiments

Each experiment discovers the structural MIS universe first and then evaluates linear/Pareto evidence only for the candidate selected by the canonical `structural_coverage` ranking. The common comparison score is `global_standardized_external_r2`.

In [ ]:
comparative_results = {}
comparison_rows = []

for case_id, generator in COMPARATIVE_CASES:
    data, truth = generator(N=N, seed=SEED)
    mis_set = misda.discover(data, name=truth["name"], seed=SEED)
    structural = misda.rank(mis_set)
    misda.evaluate(mis_set, metrics=("linear", "pareto"), candidates=structural[:1])
    benchmark_result = misda.benchmark(mis_set, truth)
    print(benchmark_result.report())
    mis_set.graph_plot(ranking=structural)

    misda_common = bench.misda_global_standardized_external_r2(data, mis_set)
    pca_external_curve = bench.pca_external_reconstruction_curve(data, max_components=data.shape[1])
    selected_dimension = structural.selected_dimension
    pca_same_dimension = next(
        point[bench.COMMON_RECONSTRUCTION_METRIC]
        for point in pca_external_curve
        if point["dimension"] == selected_dimension
    )
    pca_native_curve = bench.pca_in_sample_reconstruction_curve(data, max_components=min(10, data.shape[1]))

    comparative_results[case_id] = {
        "result_obj": mis_set,
        "ranking": structural,
        "benchmark_obj": benchmark_result,
        "truth": truth,
        "misda_common": misda_common,
        "pca_external_curve": pca_external_curve,
        "pca_native_curve": pca_native_curve,
    }
    comparison_rows.append({
        "case_id": case_id,
        "name": truth["name"],
        "dimension": selected_dimension,
        "misda_global_standardized_external_r2": misda_common,
        "pca_global_standardized_external_r2": pca_same_dimension,
        "misda_minus_pca": misda_common - pca_same_dimension,
    })

## Direct comparison at the MISDA-selected dimension

In [ ]:
comparison = pd.DataFrame(comparison_rows)
comparison

## MISDA native reconstruction diagnostics

These summarize only objectives eliminated by the selected MIS and are not directly compared with PCA's native score.

In [ ]:
misda_reconstruction = pd.DataFrame(
    {
        "case_id": case_id,
        "name": item["truth"]["name"],
        "selected_dimension": item["ranking"].selected_dimension,
        "mean_eliminated_objective_r2": item["ranking"].selected.linear.mean_r2,
        "worst_eliminated_objective_r2": item["ranking"].selected.linear.worst_r2,
    }
    for case_id, item in comparative_results.items()
)
misda_reconstruction

## PCA native in-sample reconstruction

In [ ]:
pca_reconstruction = pd.DataFrame(
    {
        "case_id": case_id,
        "name": item["truth"]["name"],
        "dimension": point["dimension"],
        "global_standardized_r2": point["global_standardized_r2"],
    }
    for case_id, item in comparative_results.items()
    for point in item["pca_native_curve"]
)
pca_reconstruction